# Lab 2: Matrix Transpose

---

## Overview

This lab introduces **2D thread indexing** and the critical concept of **memory coalescing** through matrix transpose. Matrix transpose is a fundamental operation in linear algebra that reveals the performance implications of memory access patterns on GPUs.

Unlike vector addition where memory access is naturally coalesced, matrix transpose creates a conflict: reading rows (coalesced) requires writing columns (non-coalesced), or vice versa. This lab teaches you to use **shared memory tiling** to achieve coalesced access in both read and write operations.

Building on Lab 1's 1D indexing, this lab extends your skills to 2D problems—essential for image processing, matrix operations, and neural networks.

## Learning Objectives

By the end of this lab, you will be able to:

1. Use 2D thread and block indexing (`blockIdx.x/y`, `threadIdx.x/y`)
2. Understand row-major memory layout and linear index calculation
3. Explain why naive transpose has non-coalesced memory writes
4. Analyze the performance impact of coalesced vs. strided access
5. Implement shared memory tiling to optimize memory access patterns

## 1. Matrix Transpose Problem

### Mathematical Definition

Given an input matrix $A$ of size $(rows \times cols)$, the transpose $A^T$ has size $(cols \times rows)$ where:

$$
A^T[j][i] = A[i][j]
$$

### Example

```
Input (3×4):          Output (4×3):
┌─────────────────┐   ┌───────────────┐
│  1   2   3   4  │   │  1   5   9    │
│  5   6   7   8  │ → │  2   6  10    │
│  9  10  11  12  │   │  3   7  11    │
└─────────────────┘   │  4   8  12    │
                      └───────────────┘
```

## 2. Row-Major Memory Layout

In C/C++, matrices are stored in **row-major order**: elements of each row are contiguous in memory.

### Memory Layout Example

For a 3×4 matrix:

```
Logical View:           Memory Layout:
┌─────────────────┐     
│  1   2   3   4  │     Address: [0] [1] [2] [3] [4] [5] [6] [7] [8] [9] [10] [11]
│  5   6   7   8  │  →  Value:    1   2   3   4   5   6   7   8   9  10   11   12
│  9  10  11  12  │     
└─────────────────┘     Row 0      |   Row 1       |   Row 2
```

### Index Calculation

For element at position $(i, j)$ in a matrix with `cols` columns:

$$
\text{linear\_index} = i \times cols + j
$$

## 3. 2D Thread Indexing

For matrix operations, we use 2D grids and blocks:

```cpp
dim3 threadsPerBlock(BLOCK_SIZE, BLOCK_SIZE);  // e.g., (32, 32)
dim3 blocksPerGrid((cols + BLOCK_SIZE - 1) / BLOCK_SIZE,
                   (rows + BLOCK_SIZE - 1) / BLOCK_SIZE);
```

### Global Index Calculation

```cpp
int idx = blockIdx.x * blockDim.x + threadIdx.x;  // Column index
int idy = blockIdx.y * blockDim.y + threadIdx.y;  // Row index
```

### Visual Example (4×4 grid with 2×2 blocks)

```
             Block (0,0)    Block (1,0)
           ┌───────────┐ ┌───────────┐
Row 0      │(0,0) (1,0)│ │(2,0) (3,0)│
Row 1      │(0,1) (1,1)│ │(2,1) (3,1)│
           └───────────┘ └───────────┘
             Block (0,1)    Block (1,1)
           ┌───────────┐ ┌───────────┐
Row 2      │(0,2) (1,2)│ │(2,2) (3,2)│
Row 3      │(0,3) (1,3)│ │(2,3) (3,3)│
           └───────────┘ └───────────┘
           (idx, idy) = (column, row)
```

### Exercise 1: 2D Indexing

Given `blockDim = (32, 32)`, calculate the global indices:

| blockIdx | threadIdx | Global (idx, idy) |
|:---------|:----------|:------------------|
| (0, 0) | (5, 10) | (5, 10) |
| (1, 0) | (0, 0) | (32, 0) |
| (2, 3) | (15, 20) | (79, 116) |

**Question**: For a matrix with `rows=100, cols=200`, how many blocks are needed?

Your calculation: `ceil(200/32) = 7` blocks in x and `ceil(100/32) = 4` blocks in y, so `blocksPerGrid = (7, 4)` and the total number of blocks is `7 * 4 = 28`.

## 4. Naive Transpose Kernel

Let us examine the kernel in `main.hip`:

```cpp
#define BLOCK_SIZE 32

__global__ void matrix_transpose_kernel(const float* input, float* output, 
                                        int rows, int cols) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;  // Column
    int idy = blockIdx.y * blockDim.y + threadIdx.y;  // Row

    if (idx < cols && idy < rows) {
        // Read from input[idy][idx], write to output[idx][idy]
        output[idx * rows + idy] = input[idy * cols + idx];
    }
}
```

### Index Mapping

| Operation | Matrix Notation | Linear Index |
|:----------|:----------------|:-------------|
| Read | `input[idy][idx]` | `idy * cols + idx` |
| Write | `output[idx][idy]` | `idx * rows + idy` |

### Exercise 2: Understand the Index Transformation

For a 4×6 matrix (rows=4, cols=6), fill in the table:

| Thread (idx, idy) | Input Index | Output Index |
|:------------------|:------------|:-------------|
| (0, 0) | 0*6+0 = 0 | 0*4+0 = 0 |
| (1, 0) | 0*6+1 = 1 | 1*4+0 = 4 |
| (0, 1) | 1*6+0 = 6 | 0*4+1 = 1 |
| (3, 2) | 2*6+3 = 15 | 3*4+2 = 14 |

**Question**: Why is the write index formula `idx * rows + idy` instead of `idx * cols + idy`?

Your answer: The output matrix has shape `cols × rows`, so each output row contains `rows` elements. In row-major order, `output[idx][idy]` is therefore stored at `idx * rows + idy`. Using `idx * cols + idy` would incorrectly use the input row length, and it gives wrong positions whenever the matrix is not square.

## 5. Setup: Generate Test Data

In [ ]:
%%bash
# Generate test cases
python3 geninput.py
echo "Test cases generated:"
ls -la testcases/

## 6. Compile the Program

In [ ]:
%%bash
# Recompile (now with HIP-event kernel timing printed to stderr)
hipcc -O2 fs_main.hip -o exe_transpose
echo "Compilation complete."

## 7. Run Test Cases & Multi-Trial Benchmark

First a correctness check, then a multi-trial benchmark using **true kernel time** measured by HIP events inside the program (printed on stderr as `KERNEL_MS <ms>`).

This excludes file I/O, process startup, `std::cout` printing, and H2D/D2H copies — so the bandwidth numbers reflect the GPU memory subsystem, not the OS.

In [ ]:
%%bash
echo "=== Correctness check: Small matrix (16x16) ==="
head -3 testcases/1.in
echo "..."
./exe_transpose testcases/1.in | head -3

In [ ]:
import subprocess, re, statistics

# Multi-trial benchmark — parse real kernel time from stderr (HIP events)
TRIALS = 10
KERN_RE = re.compile(r"KERNEL_MS\s+([0-9.eE+-]+)")
test_cases = [
    ("Test 1: 16x16",       "testcases/1.in",       16,   16),
    ("Test 2: 128x32",      "testcases/2.in",      128,   32),
    ("Test 3: 1x1024",      "testcases/3.in",        1, 1024),
    ("Test 4: 1001x2001",   "testcases/4.in",     1001, 2001),
]

results = {}
for name, path, rows, cols in test_cases:
    kernel_times = []
    for _ in range(TRIALS):
        r = subprocess.run(["./exe_transpose", path],
                           capture_output=True, text=True)
        m = KERN_RE.search(r.stderr)
        if not m:
            raise RuntimeError(f"No KERNEL_MS in stderr for {path}: {r.stderr[:200]}")
        kernel_times.append(float(m.group(1)))
    n_elem = rows * cols
    results[name] = (n_elem, kernel_times)

print(f"{'Test':<22} {'Elements':>10} {'Mean (ms)':>12} {'Std (ms)':>10} {'Min':>10} {'Max':>10}")
print("-" * 80)
for name, (n_elem, t) in results.items():
    m, s = statistics.mean(t), statistics.stdev(t) if len(t) > 1 else 0.0
    print(f"{name:<22} {n_elem:>10} {m:>12.4f} {s:>10.4f} {min(t):>10.4f} {max(t):>10.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Plot: real kernel time (us) and per-element cost
names = list(results.keys())
elems = [results[n][0] for n in names]
means_ms = [statistics.mean(results[n][1]) for n in names]
stds_ms  = [statistics.stdev(results[n][1]) if len(results[n][1]) > 1 else 0.0 for n in names]
means_us = [m * 1000 for m in means_ms]
stds_us  = [s * 1000 for s in stds_ms]
per_elem_ns = [(m * 1e6) / e for m, e in zip(means_ms, elems)]  # ns per element

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

short_names = [n.split(":")[0] for n in names]

# Left: kernel time in microseconds
ax1.bar(short_names, means_us, yerr=stds_us, capsize=5,
        color=['#4c72b0', '#dd8452', '#55a467', '#c44e52'], edgecolor='black')
ax1.set_ylabel("Kernel Time (us)")
ax1.set_yscale('log')
ax1.set_title(f"Naive Transpose Kernel Time — HIP events ({TRIALS} trials)")
ax1.grid(axis='y', alpha=0.3, which='both')
for i, m in enumerate(means_us):
    ax1.text(i, m, f"{m:.2f}", ha='center', va='bottom', fontsize=9)

# Right: per-element cost (ns) — exposes launch overhead
ax2.bar(short_names, per_elem_ns,
        color=['#4c72b0', '#dd8452', '#55a467', '#c44e52'], edgecolor='black')
ax2.set_yscale('log')
ax2.set_ylabel("Time per Element (ns, log)")
ax2.set_title("Per-Element Cost\n(small tests dominated by kernel launch overhead)")
ax2.grid(axis='y', alpha=0.3, which='both')
for i, v in enumerate(per_elem_ns):
    ax2.text(i, v, f"{v:.2f}", ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig("transpose_kernel_time.png", dpi=150)
plt.show()
print("Saved: transpose_kernel_time.png")

In [ ]:
# Effective bandwidth = (read + write bytes) / kernel_time
# Now using REAL kernel time from HIP events (not wall time).

bytes_per_elem = 4  # float
bw = []
for n in names:
    e, t = results[n]
    mean_s = statistics.mean(t) / 1000.0  # ms -> s
    total_bytes = 2 * e * bytes_per_elem  # read + write
    bw.append(total_bytes / mean_s / 1e9)  # GB/s

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(short_names, bw,
              color=['#4c72b0', '#dd8452', '#55a467', '#c44e52'],
              edgecolor='black')
ax.set_ylabel("Effective Bandwidth (GB/s)")
ax.set_title("Naive Transpose Effective Bandwidth (real kernel time)\n"
             "small tests: launch overhead dominates; large test: strided writes hurt")
ax.axhline(256, color='red', linestyle='--', alpha=0.6,
           label="Radeon 8060S LPDDR5x peak ≈ 256 GB/s")
ax.grid(axis='y', alpha=0.3)
ax.legend()
for b, v in zip(bars, bw):
    ax.text(b.get_x() + b.get_width()/2, v, f"{v:.1f}",
            ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig("transpose_bandwidth.png", dpi=150)
plt.show()

print("\n=== Interpretation ===")
print("- Small tests (16x16, 1x1024): only a handful of wavefronts, launch overhead dominates")
print("- Large test (1001x2001): real memory behavior visible; writes are STRIDED")
print("- Gap between achieved BW and 256 GB/s peak == cost of non-coalesced writes")
print("- A tiled transpose using LDS would close most of this gap")

### Exercise 3: Performance Analysis

Record execution times:

| Test | Dimensions | Elements | Time |
|:-----|:-----------|:---------|:-----|
| 1 | 16×16 | 256 | 0.0077 ms |
| 2 | 128×32 | 4,096 | 0.0108 ms |
| 3 | 1×1024 | 1,024 | 0.0087 ms |
| 4 | 1001×2001 | 2,003,001 | 0.2352 ms |

**Question**: The large test has ~500x more elements than test 2. Is the time 500x longer? Why or why not?

Your answer: No. In the measured results, Test 4 has about `2,003,001 / 4,096 = about 489` times as many elements as Test 2, but its time is only about `0.2352 / 0.0108 = about 21.8` times longer. Small cases are dominated by fixed costs such as kernel launch overhead, scheduling, and low occupancy, so their per-element time is relatively high. The large case exposes more parallelism and amortizes those fixed costs, but the naive transpose still writes with a large stride, causing non-coalesced global-memory transactions. Therefore the runtime is affected by occupancy, memory bandwidth, cache behavior, and non-coalesced write transactions rather than by element count alone.

## 8. Memory Coalescing Analysis

### What is Memory Coalescing?

When threads in a wavefront access **consecutive memory addresses**, the GPU can combine these into a single memory transaction. This is called **coalesced access**.

> **Note**: This lab targets **RDNA 3.5 (wave32)**. A wavefront here is 32 threads. On CDNA (MI100/MI210), it would be 64.

### Coalesced vs Non-Coalesced Access

| Access Pattern | Transactions | Bandwidth Utilization |
|:---------------|:-------------|:----------------------|
| Coalesced (consecutive) | 1 per wavefront | High (100%) |
| Strided | Multiple | Low (can be <10%) |
| Random | 1 per thread | Very low |

### Transpose Memory Access Pattern

Consider threads with consecutive `idx` values (same row of threads):

```
Thread idx:    0      1      2      3     ... (consecutive in x)
Read from:  [idy*cols+0] [idy*cols+1] [idy*cols+2] ...  → CONSECUTIVE (coalesced)
Write to:   [0*rows+idy] [1*rows+idy] [2*rows+idy] ...  → STRIDED by 'rows' (non-coalesced)
```

### Visualization

```
READ (Row-major, coalesced):     WRITE (Column-major, strided):
┌──────────────────────┐         Memory addresses:
│ T0→ T1→ T2→ T3→ ... │         T0 writes to addr 0
│                      │         T1 writes to addr rows (e.g., 1000)
│                      │         T2 writes to addr 2*rows (e.g., 2000)
└──────────────────────┘         ...
  Consecutive addresses          Strided by 'rows' apart!
```

### Exercise 4: Coalescing Analysis

For a 1024×1024 matrix with 32×32 thread blocks (one wave32 = one row of the block):

**Question 1**: When reading `input[idy * cols + idx]`, are consecutive threads (in `threadIdx.x`) reading consecutive addresses?

Your answer: Yes. Consecutive threads in `threadIdx.x` have consecutive `idx` values while `idy` is fixed, so they read `input[idy * cols + idx]` from consecutive float addresses. This read is coalesced.

**Question 2**: When writing `output[idx * rows + idy]`, what is the stride between addresses written by consecutive threads?

Your answer: The stride is `rows = 1024` floats between consecutive threads, i.e. `1024 * 4 = 4096` bytes.

**Question 3**: If each memory transaction fetches 128 bytes (32 floats), how many transactions are needed for 32 threads (one wavefront) to write non-coalesced?

Your answer: 32 transactions. With a 4096-byte stride, each of the 32 threads writes to a different 128-byte memory segment, so the wavefront cannot combine the writes into one coalesced transaction.

## 9. Optimized Transpose with Shared Memory

### The Tiling Strategy

Use shared memory as a staging area:

1. **Read tile coalesced** from input into shared memory
2. **Transpose in shared memory** (fast, no coalescing concern)
3. **Write tile coalesced** from shared memory to output

```cpp
__global__ void transpose_tiled(const float* input, float* output,
                                 int rows, int cols) {
    __shared__ float tile[BLOCK_SIZE][BLOCK_SIZE + 1];  // +1 to avoid bank conflicts
    
    int x = blockIdx.x * BLOCK_SIZE + threadIdx.x;
    int y = blockIdx.y * BLOCK_SIZE + threadIdx.y;
    
    // Read coalesced (row-major)
    if (x < cols && y < rows)
        tile[threadIdx.y][threadIdx.x] = input[y * cols + x];
    __syncthreads();
    
    // Calculate output position (transposed block)
    x = blockIdx.y * BLOCK_SIZE + threadIdx.x;
    y = blockIdx.x * BLOCK_SIZE + threadIdx.y;
    
    // Write coalesced (row-major in output)
    if (x < rows && y < cols)
        output[y * rows + x] = tile[threadIdx.x][threadIdx.y];
}
```

### Why This Works

```
Step 1: Load tile (coalesced read)
┌─────────────────┐          ┌─────────────────┐
│ Input (global)  │  ─────>  │ Shared Memory   │
│ Row-major read  │          │ tile[y][x]      │
└─────────────────┘          └─────────────────┘

Step 2: Write transposed tile (coalesced write)
┌─────────────────┐          ┌─────────────────┐
│ Shared Memory   │  ─────>  │ Output (global) │
│ tile[x][y]      │          │ Row-major write │
└─────────────────┘          └─────────────────┘
```

| Access | Pattern | Coalesced? |
|:-------|:--------|:-----------|
| Read from input | Row-major | ✓ Yes |
| Write to shared | Any (fast) | N/A |
| Read from shared | Transposed | N/A |
| Write to output | Row-major | ✓ Yes |

### Exercise 5: Tiled Transpose Analysis

**Question 1**: Why do we add `+1` to the shared memory declaration (`tile[BLOCK_SIZE][BLOCK_SIZE + 1]`)?

Your answer: The `+1` padding changes the shared-memory row pitch from 32 floats to 33 floats. When the tile is accessed in transposed order, this prevents all threads from hitting the same shared-memory bank, avoiding severe bank conflicts.

**Question 2**: In the write step, why do we swap blockIdx.x and blockIdx.y?

Your answer: The input tile at block `(blockIdx.x, blockIdx.y)` becomes an output tile at `(blockIdx.y, blockIdx.x)` after transpose. Swapping the block coordinates makes each block write the transposed tile to the correct output position while keeping global writes coalesced.

**Question 3**: How much shared memory does each block use for BLOCK_SIZE=32?

Your calculation: `32 * (32 + 1) * sizeof(float) = 32 * 33 * 4 = 4224` bytes.

## 10. Bandwidth Analysis

### Theoretical vs Achieved Bandwidth

For an `N × N` float transpose:
- **Data transferred** = `2 × N × N × 4` bytes (one read + one write)
- **Achieved bandwidth** = data / kernel time

**Reference peaks (AMD Radeon 8060S iGPU, gfx1151, RDNA 3.5):**

| Memory | Peak (approx.) |
|:-------|:---------------|
| LPDDR5x system memory | **~256 GB/s** |
| L2 cache | much higher |

> Verify on your system with `rocminfo` (look for memory bandwidth). For comparison, MI100 HBM2 peaks at ~1.2 TB/s; this iGPU shares LPDDR5x with the CPU.

### Why is naive transpose far below peak?

The plot above shows the naive kernel falling well short of 256 GB/s on the large test. The cause is the **strided write** pattern: each wave32 produces 32 writes to addresses separated by `rows` floats apart, so each write costs its own memory transaction instead of one coalesced 128-byte burst.

## 11. Summary

### Key Concepts

| Concept | Description |
|:--------|:------------|
| **2D Indexing** | Using `blockIdx.x/y` and `threadIdx.x/y` for matrix operations |
| **Row-Major Layout** | C/C++ stores matrices with rows contiguous in memory |
| **Memory Coalescing** | Consecutive threads accessing consecutive addresses |
| **Shared Memory Tiling** | Using shared memory to enable coalesced access |

### Performance Impact of Coalescing

| Pattern | Memory Transactions | Relative Performance |
|:--------|:--------------------|:---------------------|
| Coalesced | 1 per warp | 1.0x (baseline) |
| Strided by 1024 | 32 per warp | ~0.03x (30x slower) |

### Best Practices

| Guideline | Reason |
|:----------|:-------|
| Consecutive threads → consecutive addresses | Memory coalescing |
| Use shared memory for transpose | Decouple read/write patterns |
| BLOCK_SIZE = 32 | Matches warp size, good for coalescing |
| Pad shared memory | Avoid bank conflicts |

## 12. Challenge Exercises

### Challenge 1: Implement Tiled Transpose

Modify `main.hip` to use the shared memory tiling approach from Section 9. Compare performance with the naive version.

### Challenge 2: Bank Conflict Analysis

Test with and without the `+1` padding in shared memory. Measure the performance difference.

### Challenge 3: In-Place Transpose

For square matrices, implement in-place transpose using only O(1) extra memory.

### Challenge 4: Rectangular Tile Sizes

Experiment with non-square tile sizes (e.g., 32×8 or 16×64). When might these be beneficial?

### Challenge 5: Double-Buffering

Implement double-buffering to overlap memory transfers with computation for very large matrices.

---

## Lab Files Reference

| File | Description |
|:-----|:------------|
| `main.hip` | Naive transpose implementation (stdin input) |
| `fs_main.hip` | Naive transpose implementation (file input) |
| `geninput.py` | Test case generator |
| `Makefile` | Build configuration |